In [1]:
import os
import copy
import torch
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

NUM_CLASSES = 16
BATCH_SIZE = 16
LOCAL_EPOCHS = 15
LR = 1e-4

TRAIN_DIR = "train"
VAL_DIR = "unused_validation"
CORRUPT_VAL_DIR = "new_validation"
TEST_DIR = "test"

cuda


In [3]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


### Pre-trained Model Loading

In [4]:
# Loading ResNet-50
# def get_model():
#     model = models.resnet50(pretrained=True)
#     model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading Swin_v2_Base
# def get_model():
#     model = models.swin_v2_b(weights=models.Swin_V2_B_Weights.IMAGENET1K_V1)
#     model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading ViT B-16
def get_model():
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
    return model.to(DEVICE)

# Loading ConvNext-Base
# def get_model():
#     model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
#     model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading EfficientNet-V2-S
# def get_model():
#     model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
#     model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
#     return model.to(DEVICE)

In [5]:
def get_client_loader(client_name):
    client_path = os.path.join(TRAIN_DIR, client_name)
    print(f"Loading data from {client_path}")
    dataset = datasets.ImageFolder(client_path, transform=train_transform)
    print(f"Number of training data instances: {len(dataset)}")
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [6]:
val_loader = DataLoader(
    datasets.ImageFolder(VAL_DIR, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)

corrupt_val_loader = DataLoader(
    datasets.ImageFolder(CORRUPT_VAL_DIR, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    datasets.ImageFolder(TEST_DIR, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)

### Training & Validation - Client

In [7]:
def local_train(model, loader):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(LOCAL_EPOCHS):
        epoch_loss = 0.0
        progress_bar = tqdm(loader, desc=f"Local Epoch [{epoch+1}/{LOCAL_EPOCHS}]", leave=False)

        for x, y in progress_bar:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        print(f"Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    return model.state_dict()

In [8]:
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro')  
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    return acc, precision, recall, f1

In [9]:
clients = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d)) and not d.startswith(".")
])

print(f"Found clients: {clients}")

Found clients: ['client_1', 'client_2', 'client_3', 'client_4', 'client_5']


In [10]:
CKPT_DIR = "ViT-B-16"      # Checkpoint Directory
os.makedirs(CKPT_DIR, exist_ok=True)

client_models = []
corrupt_client_val_accs = []
client_val_accs = []
client_val_f1s = []

for client in clients:
    client_loader = get_client_loader(client)
    model = get_model()

    model_dict = local_train(model, client_loader)

    client_val_acc, client_val_pre, client_val_recall, client_val_f1 = evaluate(model, val_loader)
    corrupt_client_val_acc, corrupt_client_val_pre, corrupt_client_val_recall, corrupt_client_val_f1 = evaluate(model, corrupt_val_loader) # Corrupted validation evaluation

    print(f"{client} Validation Accuracy: {client_val_acc:.4f}")
    print(f"{client} Validation Precision: {client_val_pre:.4f}")
    print(f"{client} Validation Recall: {client_val_recall:.4f}")
    print(f"{client} Validation F1 Score: {client_val_f1:.4f}")
    print("\n ============================================== \n")
    print(f"{client} Corrupted Validation Accuracy: {corrupt_client_val_acc:.4f}")
    print("\n ============================================== \n")
        
    torch.save({
        "client_name": client,
        "model_state_dict": model_dict,
        "val_accuracy": client_val_acc,
        "val_precision": client_val_pre,
        "val_recall": client_val_recall,
        "val_f1": client_val_f1
    }, os.path.join(CKPT_DIR, f"{client}.pth"))

    client_models.append(model)
    client_val_accs.append(client_val_acc)
    client_val_f1s.append(client_val_f1)
    corrupt_client_val_accs.append(corrupt_client_val_acc)

Loading data from train/client_1
Number of training data instances: 44803


Epoch 1 Loss: 1.1656


Epoch 2 Loss: 0.7377


Epoch 3 Loss: 0.5989


Epoch 4 Loss: 0.5039


...

Epoch 11 Loss: 0.1558


Epoch 12 Loss: 0.1383


Epoch 13 Loss: 0.1275


Epoch 14 Loss: 0.1125


Epoch 15 Loss: 0.1039
client_5 Validation Accuracy: 0.8077
client_5 Validation Precision: 0.8110
client_5 Validation Recall: 0.8074
client_5 Validation F1 Score: 0.8082


client_5 Corrupted Validation Accuracy: 0.7891




### ReSoFed

In [11]:
def average_models(models):
    avg_model = copy.deepcopy(models[0])
    avg_dict = avg_model.state_dict()

    for key in avg_dict.keys():
        tensors = [m.state_dict()[key] for m in models]

        if tensors[0].dtype in [torch.float32, torch.float64, torch.float16]:
            avg_dict[key] = torch.stack(tensors, dim=0).mean(dim=0)
        else:
            avg_dict[key] = tensors[0]

    avg_model.load_state_dict(avg_dict)
    return avg_model

In [12]:
def greedy_soup(models, val_loader, val_scores):
    sorted_indices = sorted(range(len(models)), key=lambda i: val_scores[i], reverse=True)
    soup_model = copy.deepcopy(models[sorted_indices[0]])
    best_score = val_scores[sorted_indices[0]]
    selected = [sorted_indices[0]]

    for idx in sorted_indices[1:]:
        temp_model = average_models([soup_model, models[idx]])
        score = evaluate(temp_model, val_loader)[0]

        if score > best_score:
            soup_model = temp_model
            best_score = score
            selected.append(idx)

    return soup_model, selected, val_scores

In [13]:
def reliability_weighted_avg(models, reliabilities, selected_indices):
    weights = torch.tensor([reliabilities[i] for i in selected_indices])
    weights = weights / weights.sum()

    global_model = copy.deepcopy(models[selected_indices[0]])
    global_dict = global_model.state_dict()

    for key in global_dict.keys():
        weighted_params = torch.stack([
            weights[i] * models[selected_indices[i]].state_dict()[key]
            for i in range(len(selected_indices))
        ], dim=0)
        
        global_dict[key] = weighted_params.sum(dim=0)

    global_model.load_state_dict(global_dict)
    return global_model

In [14]:
# Greedy Soup
soup_model, selected_indices, val_accs_clean = greedy_soup(client_models, val_loader, client_val_accs)

In [15]:
resofed_model = reliability_weighted_avg(
    client_models,
    val_accs_clean,
    selected_indices
)

In [16]:
acc, precision, recall, f1 = evaluate(resofed_model, test_loader)

print("Original ReSoFed Results:")
print(f"Global Model Test Accuracy: {acc:.4f}")
print(f"Global Model Test Precision: {precision:.4f}")
print(f"Global Model Test Recall: {recall:.4f}")
print(f"Global Model Test F1 Score: {f1:.4f}")

print("Selected Clients:", selected_indices)

Original ReSoFed Results:
Global Model Test Accuracy: 0.8114
Global Model Test Precision: 0.8175
Global Model Test Recall: 0.8110
Global Model Test F1 Score: 0.8128
Selected Clients: [1]


## ReSoFed-Doc (New)

In [17]:
import numpy as np


A_clean = np.array(client_val_accs)
A_corrupt = np.array(corrupt_client_val_accs)

robustness = A_corrupt * (A_corrupt / (A_clean + 1e-8))  # Adding small epsilon to avoid division by zero
alpha = 0.5 
reliability = alpha * A_clean + (1 - alpha) * robustness

print("Reliability Scores:", reliability)

Reliability Scores: [0.78765953 0.79596569 0.77388908 0.7814824  0.78926623]


In [18]:
global_model = reliability_weighted_avg(
    client_models,
    reliability,
    selected_indices
)

In [19]:
acc, precision, recall, f1 = evaluate(global_model, test_loader)

print("\nNew ReSoFed-Doc Results:")
print(f"Global Model Test Accuracy: {acc:.4f}")
print(f"Global Model Test Precision: {precision:.4f}")
print(f"Global Model Test Recall: {recall:.4f}")
print(f"Global Model Test F1 Score: {f1:.4f}")

print("Selected Clients:", selected_indices)


New ReSoFed-Doc Results:
Global Model Test Accuracy: 0.8114
Global Model Test Precision: 0.8175
Global Model Test Recall: 0.8110
Global Model Test F1 Score: 0.8128
Selected Clients: [1]


### FedAvg (Baseline)

In [20]:
def fedavg(models):

    global_model = copy.deepcopy(models[0])
    global_dict = global_model.state_dict()

    for key in global_dict.keys():

        tensors = [models[i].state_dict()[key] for i in range(len(models))]

        if tensors[0].dtype.is_floating_point:
            global_dict[key] = torch.stack(tensors, dim=0).mean(dim=0)
        else:
            global_dict[key] = tensors[0]

    global_model.load_state_dict(global_dict)
    return global_model

fedavg_model = fedavg(client_models)
fedavg_test_acc, fedavg_test_pre, fedavg_test_recall, fedavg_test_f1 = evaluate(fedavg_model, test_loader)
print("FedAvg Test Accuracy:", fedavg_test_acc)
print("FedAvg Test Precision:", fedavg_test_pre)
print("FedAvg Test Recall:", fedavg_test_recall)
print("FedAvg Test F1 Score:", fedavg_test_f1)

FedAvg Test Accuracy: 0.7044852242612131
FedAvg Test Precision: 0.7586009130929162
FedAvg Test Recall: 0.7043278905859255
FedAvg Test F1 Score: 0.710975094905595
